# មេរៀន 12 - ការកាត់បន្ថយប្រវត្តិការជជែកជាមួយ Agent Scratchpad

សៀវភៅកត់ត្រានេះបង្ហាញពីរបៀបគ្រប់គ្រងបរិបទក្នុងការសន្ទនាយូរដោយប្រើ Microsoft Agent Framework។ នៅពេលការសន្ទនាធំឡើង ចំនួន token កើនឡើង — និងចុងក្រោយអាចលើសបរិវេណបរិបទរបស់ម៉ូឌែល។ យើងដោះស្រាយបញ្ហានេះដោយប្រើ **ទំរង់សង្ខេបបរិបទ** និង **Agent Scratchpad** សម្រាប់ចងចាំថេរ។

## អ្វីដែលអ្នកនឹងរៀន:
1. **ហេតុអ្វីបានជា ការគ្រប់គ្រងបរិបទមានសារៈសំខាន់**: ការយល់ដឹងអំពីដែនកំណត់ token និងបរិវេណបរិបទ
2. **ភ្នាក់ងារដែលយល់ដឹងពីបរិបទ**: ការសាងសង់ភ្នាក់ងារដែលគ្រប់គ្រងបរិបទសន្ទនារបស់ខ្លួន
3. **ទំរង់សង្ខេបបរិបទ**: ការប្រើឧបករណ៍ដើម្បីសង្ខេបប្រវត្តិការសន្ទនា
4. **Agent Scratchpad**: អង្គចងចាំថេរដែលនៅសល់បន្ទាប់ពីការកាត់បន្ថយបរិបទ

## លក្ខខណ្ឌមុន:
- ការតំឡើង Azure OpenAI ជាមួយអថេរបរិយាកាសដែលបានកំណត់
- មានការយល់ដឹងទូទៅអំពីគំនិតមូលដ្ឋាននៃភ្នាក់ងារពីមេរៀនមុនៗ


## ការដំឡើង


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## ហេតុអ្វីការគ្រប់គ្រងបរិបទមានសារៈសំខាន់

Every LLM has a finite **បង្អួចបរិបទ** — the maximum number of tokens it can process in a single request. As a multi-turn conversation progresses:

- **ចំនួនតូកិនកើនឡើងដោយសមាមាត្រ** ជាមួយសារ​មួយនីមួយៗ​របស់អ្នកប្រើ និងការឆ្លើយតប​របស់ជំនួយការ។
- **តូកិន​បញ្ចូល គ្របដណ្តប់លើចំណាយ** ព្រោះប្រវត្តិទាំងមូលត្រូវបានផ្ញើឡើងវិញរៀងរាល់ជុំ។
- ជាយ៉ាងចុងក្រោយ ការសន្ទនា **លើសបង្អួចបរិបទ** និងម៉ូឌែលនឹងធ្វើការកាត់ចេញ ឬបង្ហាញកំហុស។

### យុទ្ធសាស្ត្រសម្រាប់គ្រប់គ្រងបរិបទ

| យុទ្ធសាស្ត្រ | របៀបដែលវាធ្វើការ | ការជម្លោះ |
|---|---|---|
| **ការកាត់ចេញ** | លុបសារចាស់បំផុត | បាត់បង់បរិបទដើម |
| **សង្ខេប** | បង្រួមសារចាស់ៗទៅជាសេចក្ដីសង្ខេប | ព័ត៌មានលំអិតខ្លះៗបាត់បង់ ប៉ុន្តែចំណុចសំខាន់ៗត្រូវបានរក្សាទុក |
| **Scratchpad / អង្គចងចាំខាងក្រៅ** | រក្សាព័ត៌មានសំខាន់នៅក្រៅសន្ទនា | តម្រូវឲ្យមានការ​ហៅឧបករណ៍ ប៉ុន្តែអាចទ្រាំទ្របានក្រោយការកាត់បន្ថយណាមួយ |

In this notebook we combine **សង្ខេប** with a **ឧបករណ៍អង្គចងចាំខាងក្រៅ** so the agent can maintain continuity even when conversation history is condensed.


## បង្កើតភ្នាក់ងារដែលមានការយល់ដឹងអំពីបរិបទ


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## ការសម្ដែងសន្ទនាយូរមួយ

យើងមកឆ្លងកាត់ការសន្ទនាច្រើនជុំ ដើម្បីមើលថាបរិបទត្រូវបានប្រមូលសរុបយ៉ាងដូចម្តេច។ ភ្នាក់ងារគួរតែរក្សារព័ត៌មានសំខាន់ៗ (ចំណូលចិត្ត, ថវិកា, កាលបរិច្ឆេទធ្វើដំណើរ) តាមការឆ្លើយតបក្នុងជំហានផ្សេងៗ និងបង្ហាញពីភាពបន្តនៅក្នុងសន្ទនា។


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

សូមចំណាំពីរបៀបដែលភ្នាក់ងារកាន់កាប់បរិបទពីជ្រុងមុនៗ — វាដឹងអំពីប្រទេសជប៉ុន, ស៊ូស៊ី, វត្ត, ការថតរូប, ថវិកា $3000, ការធ្វើដំណើរម្នាក់ឯង, និងរយៈពេលខែមេសា។ ក្នុងការសន្ទនាខ្លីនេះ វាធ្វើបានល្អ ប៉ុន្តែពេលដែលការសន្ទនាកើនឡើង ប្រវត្តិពេញលេញនឹងមានតម្លៃខ្ពស់ក្នុងការផ្ញើឡើងម្តងទៀត។

ចូរយើងបន្តការសន្ទនាជាមួយជ្រុងបន្ថែម ដើម្បីឃើញការប្រមូលបរិបទ៖


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## លំនាំសង្ខេបបរិបទ

ពេលការសន្ទនា​កើនឡើង យើងអាចប្រើ **ឧបករណ៍សង្ខេប** ដើម្បីបង្រួមបរិបទដែលបានប្រមូលឲ្យជា​ទ្រង់ទ្រាយសង្ខេប។ ភ្នាក់ងារ​ហៅឧបករណ៍នេះដើម្បីរក្សាទុកចំណូលចិត្តសំខាន់ៗ ដូច្នេះ ទោះបីសារចាស់ៗត្រូវបានលុបក៏ដោយ ព័ត៌មានសំខាន់ៗនឹងត្រូវបានរក្សាទុក។

លំនាំនេះជាគ្រឹះសម្រាប់ការកាត់បន្ថយប្រវត្តិដែលស្មុគស្មាញជាង:
1. ភ្នាក់ងារកំណត់កត្តាសំខាន់ៗពីការសន្ទនា
2. វាហៅឧបករណ៍សង្ខេបដើម្បីរក្សាពួកវាទុក
3. សារចាស់ៗអាចត្រូវបានដកចេញដោយសុវត្ថិភាព ព្រោះសេចក្តីសង្ខេបបានចាប់យកអ្វីដែលមានសារៈសំខាន់

ខាងក្រោម យើងកំណត់ឧបករណ៍ `summarize_preferences` មួយ ដែលភ្នាក់ងារអាចហៅដើម្បីកត់ត្រាសេចក្តីសង្ខេបខ្លីអំពីអ្វីដែលវាបានរៀន។


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## សេចក្តីសង្ខេប

ក្នុងមេរៀននេះ អ្នកបានរៀនពីរបៀបគ្រប់គ្រងបរិបទក្នុងការសន្ទនារយៈពេលវែងរបស់ភ្នាក់ងារ ដោយប្រើ Microsoft Agent Framework:

### យោគយល់សំខាន់
- **ផ្ទាំងបរិបទមានកំណត់** — សញ្ញា (token) គ្រប់មួយក្នុងប្រវត្តិការសន្ទនាសំរាប់ការប្រើប្រាស់មានការចំណាយ ហើយត្រូវបានរាប់ចូលក្នុងដែនកំណត់។
- **ឧបករណ៍​សង្ខេប** អនុញ្ញាតឲ្យភ្នាក់ងារបង្រួមបរិបទដែលបានសរុបទៅជាសេចក្តីសង្ខេបខ្លីៗ ដើម្បីកាត់បន្ថយការប្រើ token ខណៈដែលរក្សាព័ត៌មានសំខាន់ៗឲ្យនៅស្ថិត។
- **Agent scratchpads** ផ្តល់នូវចងចាំក្រៅអចិន្ដ្រៃយ៍ ដែលនៅរស់រានមានជីវិតទោះបីមានការកាត់បន្ថយការសន្ទនាណាមួយក៏ដោយ។

### អ្វីដែលអ្នកបានធ្វើ
- មួយ **ភ្នាក់ងារ​ដែលមានចំណេះដឹងពីបរិបទ** ដែលថែរក្សាបន្តភាពក្នុងការសន្ទនាច្រើនជុំ
- មួយ **ឧបករណ៍សង្ខេប** (`summarize_preferences`) ដែលកត់ត្រាព័ត៌មានសម្រួលសំខាន់របស់អ្នកប្រើជាទ្រង់ទ្រាយខ្លី
- មួយ **ការសន្ទនាច្រើនជុំ** ដែលបង្ហាញពីការរក្សាបរិបទ និងការដោះស្រាយការផ្លាស់ប្តូរ

### កម្មវិធីក្នុងជីវិតពិត
- **បូតសេវាអតិថិជន**: ចងចាំចំណង់ចំណូលចិត្តក្នុងសម័យជំនួយរយៈពេលវែង
- **ជំនួយការផ្ទាល់ខ្លួន**: តាមដានគម្រោងកំពុងដំណើរការដោយមិនចាំបាច់ពន្យល់បរិបទម្ដងទៀត
- **គ្រូបង្រៀន**: ថែរក្សាការរីកចម្រើនរបស់សិស្សឆ្លើយតបនិងអន្តរកម្មជាច្រើនដង

### ជំហានបន្ទាប់
- អនុវត្តឧបករណ៍ scratchpad ពេញលេញ ជាមួយភាពជាប់ទ្រង់ទ្រាយផ្អែកលើឯកសារ
- បន្ថែមការកាត់បន្ថយប្រវត្តិដោយស្វ័យប្រវត្តិនៅបន្ទាប់ពីការសង្ខេប
- ផ្គុំជាមួយមូលដ្ឋានទិន្នន័យវ៉ិចទ័រសម្រាប់ស្វែងរកចងចាំផ្នែកអត្ថន័យ
- សាងសង់ភ្នាក់ងារដែលអាចបន្តការសន្ទនាបានវិញពេលប៉ុន្មានថ្ងៃក្រោយជាមួយបរិបទពេញលេញ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ បើទោះយ៉ាងណាហើយ យើងខិតខំស្វែងរកភាពត្រឹមត្រូវ សូមចងចាំថាការបកប្រែដោយស្វ័យប្រវត្តិនេះអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមនៅក្នុងភាសាដើមគួរត្រូវបានគេយកគិតថាជាប្រភពផ្លូវការ។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងសូមណែនាំឲ្យប្រើការបកប្រែដោយអ្នកជំនាញមនុស្ស។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកសេចក្តីខុសដែលកើតមានពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
